# MMCultureQA — Task 2 (Textual Visual QA) baseline

**SemEval 2027 · [task website](https://mmcultureqa-semeval27.github.io/) · [data](https://huggingface.co/datasets/QCRI/MMCQA-SemEval27) · [starter kit](https://github.com/mmcultureqa-semeval27/mmcultureqa)**

Runs end to end on a free Colab T4: pulls the English **dev** split, answers each
question with **Qwen2.5-Omni-3B** in 4-bit, writes `prediction_qa_en.jsonl`, and
scores it with the official scorer — dev has reference answers, so you get a real
number before you ever submit.

For Task 1 (the spoken questions), change `TASK` to `"sqa"` in the config cell:
the audio archive is fetched and the clip goes into the prompt instead of the text.

**Runtime → Change runtime type → T4 GPU** before you start.

## 1. Install

In [ ]:
%pip install -q -U "transformers>=4.52.0" accelerate bitsandbytes qwen-omni-utils torchvision soundfile librosa datasets "huggingface_hub[hf_transfer]" pyarrow
# torchvision is needed even here: the Omni processor loads a video processor at construction
print("installed — if Colab asks you to restart the runtime, do it, then run from the next cell")


## 2. Config

In [ ]:
TASK   = "qa"        # "qa" = Task 2 (written question) | "sqa" = Task 1 (spoken)
TRACK  = "en"        # en | msa | arz | ajp
SPLIT  = "dev"       # dev has answers, so you can score yourself; devtest is the leaderboard split
LIMIT  = 100         # None = the whole split (1,000 items); 100 is a ~15 min run on a T4

MODEL        = "Qwen/Qwen2.5-Omni-3B"   # 7B: "Qwen/Qwen2.5-Omni-7B"
QUANTIZE     = True                      # 4-bit NF4, so the 3B fits a 16 GB T4
MAX_NEW      = 64                        # answers are one or two sentences
MAX_PIXELS   = 768 * 28 * 28             # lower this if you run out of memory
ROOT         = "mmcqa"                   # where the media is unpacked
OUT          = f"prediction_{TASK}_{TRACK}.jsonl"

REPO = "QCRI/MMCQA-SemEval27"
CONFIG = f"{TASK}_mena_{TRACK}"
print(f"{CONFIG} / {SPLIT} -> {OUT}")

## 3. Data

Rows come from the Hub; the media ships as zip archives that unpack into exactly the
paths the rows point at (`images/<id>.jpg`, `audio/<lang>/<id>.wav`).

In [ ]:
import os, zipfile
from datasets import load_dataset
from huggingface_hub import hf_hub_download

rows = load_dataset(REPO, CONFIG, split=SPLIT).to_list()
if LIMIT:
    rows = rows[:LIMIT]

archives = [f"images_mena_{SPLIT}.zip"]
if TASK == "sqa":
    archives.append(f"audio_mena_{TRACK}_{SPLIT}.zip")
for name in archives:
    path = hf_hub_download(REPO, f"archives/{name}", repo_type="dataset")
    with zipfile.ZipFile(path) as zf:
        zf.extractall(ROOT)
    print("unpacked", name)

missing = [r for r in rows if not os.path.isfile(os.path.join(ROOT, r["image"]))]
assert not missing, f"{len(missing)} images missing under {ROOT}"
print(f"{len(rows)} items ready | fields: {list(rows[0])}")
rows[0]

## 4. Model

In [ ]:
import torch
from transformers import (BitsAndBytesConfig, Qwen2_5OmniForConditionalGeneration,
                          Qwen2_5OmniProcessor)

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_compute_dtype=dtype,
                           bnb_4bit_use_double_quant=True) if QUANTIZE else None

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL, max_pixels=MAX_PIXELS)
model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    MODEL, torch_dtype=dtype, device_map="auto", quantization_config=quant,
    attn_implementation="sdpa")
model.disable_talker()      # text answers only; no speech is generated
model.eval()
print(f"{MODEL} loaded | 4-bit: {QUANTIZE} | dtype {dtype}")

## 5. Prompt

Qwen2.5-Omni expects its own system identity; the task instruction is appended to it.
This is a deliberately plain prompt — improving it is the first thing worth trying.

In [ ]:
from qwen_omni_utils import process_mm_info

SYSTEM = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of "
          "perceiving auditory and visual inputs, as well as generating text and speech.")
LANGUAGE = {"en": "English", "msa": "Modern Standard Arabic",
            "arz": "Egyptian Arabic", "ajp": "Levantine Arabic"}
INSTRUCTION = ("Answer the question about the image in one or two sentences, in {language}. "
               "Give only the answer.")

def answer(row):
    content = [{"type": "image", "image": os.path.join(ROOT, row["image"])}]
    if TASK == "sqa":
        content.append({"type": "audio", "audio": os.path.join(ROOT, row["audio"])})
    else:
        content.append({"type": "text", "text": row["question"]})
    content.append({"type": "text",
                    "text": INSTRUCTION.format(language=LANGUAGE.get(TRACK, "the question's language"))})
    conversation = [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
                    {"role": "user", "content": content}]

    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    audios, images, videos = process_mm_info(conversation, use_audio_in_video=False)
    inputs = processor(text=text, audio=audios, images=images, videos=videos,
                       return_tensors="pt", padding=True, use_audio_in_video=False)
    inputs = inputs.to(model.device).to(model.dtype)
    with torch.no_grad():
        generated = model.generate(**inputs, return_audio=False, do_sample=False,
                                   max_new_tokens=MAX_NEW, use_audio_in_video=False)
    out = processor.batch_decode(generated[:, inputs["input_ids"].shape[1]:],
                                 skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return out[0].strip()

print(answer(rows[0]))
print("\nquestion:", rows[0]["question"])
print("gold    :", rows[0].get("answer", "(not in this split)"))

## 6. Run and save

Answers are written to disk as they are produced, and the cell skips ids it already has,
so a disconnected runtime costs you only the item in flight. Re-run it to continue.

In [ ]:
import json, time

done = {}
if os.path.exists(OUT):
    with open(OUT, encoding="utf-8") as fh:
        done = {json.loads(l)["id"]: json.loads(l)["answer"] for l in fh if l.strip()}
    print(f"resuming: {len(done)} answers already in {OUT}")

todo = [r for r in rows if r["id"] not in done]
started = time.time()
with open(OUT, "a", encoding="utf-8") as fh:
    for i, row in enumerate(todo, 1):
        try:
            text = answer(row)
        except Exception as exc:                 # one bad item must not end the run
            print(f"  [{row['id'][:12]}] {type(exc).__name__}: {str(exc)[:90]}")
            text = ""
        fh.write(json.dumps({"id": row["id"], "answer": text, "lang": TRACK},
                            ensure_ascii=False) + "\n")
        fh.flush()
        if i % 10 == 0 or i == len(todo):
            rate = i / (time.time() - started)
            print(f"  {i}/{len(todo)}  {rate:.2f} items/s  eta {(len(todo)-i)/rate/60:.0f} min",
                  flush=True)
print(f"\nwrote {OUT}")

### A few answers, next to the references

In [ ]:
gold = {r["id"]: r.get("answer", "") for r in rows}
question = {r["id"]: r.get("question", "") for r in rows}
with open(OUT, encoding="utf-8") as fh:
    preds = [json.loads(l) for l in fh if l.strip()][:5]
for p in preds:
    print("Q    :", question.get(p["id"], "")[:100])
    print("model:", p["answer"][:150])
    print("gold :", gold.get(p["id"], "")[:150], "\n")

## 7. Score yourself

`dev` carries reference answers, so the official scorer gives you the number the
leaderboard would. It runs the challenge's own evaluation script, with the same
encoder (mmBERT, pinned revision) — BERTScore-F1 is the ranking metric.

In [ ]:
!git clone -q https://github.com/mmcultureqa-semeval27/mmcultureqa.git 2>/dev/null || git -C mmcultureqa pull -q
%pip install -q bert_score==0.3.12 sacrebleu==2.5.1 rouge-score sentencepiece

# gold = the dev rows we already loaded
with open(f"gold_{SPLIT}_{TRACK}.jsonl", "w", encoding="utf-8") as fh:
    for r in rows:
        fh.write(json.dumps({"id": r["id"], "answer": r["answer"]}, ensure_ascii=False) + "\n")

!python mmcultureqa/format_checker/check_format.py --pred {OUT} --lang {TRACK} --ids gold_{SPLIT}_{TRACK}.jsonl
!python mmcultureqa/scorer/score.py --gold gold_{SPLIT}_{TRACK}.jsonl --pred {OUT} --lang {TRACK}

## 8. Submit

Point `SPLIT` at `devtest` and run again: that split has no answers, and it is what the
development leaderboard scores. Upload `prediction_qa_en.zip` and pick the track in the
**Language** field of the submission form — one track per submission.

In [ ]:
import zipfile, shutil
shutil.copy(OUT, "prediction.jsonl")
with zipfile.ZipFile(OUT.replace(".jsonl", ".zip"), "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("prediction.jsonl")
print("ready to upload:", OUT.replace(".jsonl", ".zip"))

# keep the predictions after the runtime dies:
# from google.colab import files; files.download(OUT.replace(".jsonl", ".zip"))
# or mount Drive and copy them there:
# from google.colab import drive; drive.mount("/content/drive")
# shutil.copy(OUT, "/content/drive/MyDrive/")